In [1]:
import os
import time
import math
import random
import urllib.error
import urllib.request
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from concurrent.futures import ThreadPoolExecutor

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Funciones

In [2]:
def download_image(url, output_dir, retries=3, wait=10):
    filename = url.strip().split("/")[-1].split("?")[0]
    output_path = output_dir / filename

    # Si ya existe, verificar tamaño
    if output_path.exists():
        try:
            with Image.open(output_path) as img:
                width, height = img.size
                if width <= 161 and height <= 81:
                    print(f"🗑 Imagen ya existente inválida (tamaño {width}x{height}) → Eliminada: {filename}")
                    output_path.unlink()
                else:
                    print(f"⏩ Imagen ya existente válida: {filename}")
                return  # Ya estaba descargada, no se vuelve a descargar
        except UnidentifiedImageError:
            print(f"⚠️ Imagen corrupta (ya existente) eliminada: {filename}")
            output_path.unlink()
            return
        except Exception as e:
            print(f"⚠️ Error al verificar imagen ya existente: {filename} → {e}")
            output_path.unlink()
            return

    # Si no existe, descargar imagen
    for attempt in range(retries):
        try:
            req = urllib.request.Request(
                url.strip(),
                headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
            )
            with urllib.request.urlopen(req, timeout=10) as response, open(output_path, 'wb') as out_file:
                out_file.write(response.read())

            # Verificar tamaño después de descargar
            try:
                with Image.open(output_path) as img:
                    width, height = img.size
                    if width <= 161 and height <= 81:
                        print(f"🗑 Imagen descargada inválida (tamaño {width}x{height}) → Eliminada: {filename}")
                        output_path.unlink()
                        return
            except UnidentifiedImageError:
                print(f"⚠️ Imagen corrupta descargada → Eliminada: {filename}")
                output_path.unlink()
                return
            except Exception as e:
                print(f"⚠️ Error al verificar imagen descargada: {filename} → {e}")
                output_path.unlink()
                return

            print(f"✅ Descargada y válida: {filename}")
            time.sleep(1.5)
            break  # Salir del bucle si fue exitosa

        except urllib.error.HTTPError as e:
            if e.code == 429 and attempt < retries - 1:
                print(f"⏳ HTTP 429 Too Many Requests — esperando {wait} segundos antes de reintentar ({attempt+1}/{retries})...")
                time.sleep(wait)
            else:
                print(f"⚠️ HTTP error al descargar {url.strip()}: {e}")
                break
        except Exception as e:
            print(f"⚠️ Error general al descargar {url.strip()}: {e}")
            break


## Descargar Img

In [3]:
# === RUTAS PERSONALIZADAS ===
BASE_DIR = Path("/content/drive/MyDrive/LimpIA/data")  # Carpeta destino de imágenes
URLS_DIR = Path("/content/drive/MyDrive/LimpIA/data_url")      # Carpeta con archivos de URLs

# === CATEGORÍAS DEL DATASET ===
# class_names = ["neutral", "sexy"]
class_names = ["neutral"]

In [ ]:
# === PROCESO DE DESCARGA POR CLASE ===
for cname in class_names:
    urls_file = URLS_DIR / cname / f"urls_{cname}.txt"
    images_dir = BASE_DIR / cname / "img"
    images_dir.mkdir(parents=True, exist_ok=True)

    if urls_file.exists():
        with open(urls_file, "r") as f:
            urls = [line.strip() for line in f if line.strip()]  # Evita líneas vacías

        print(f"\n📂 Clase: {cname} — Total URLs: {len(urls)}")
        print("📥 Descargando imágenes...")

        # Descargar en paralelo pero controlado para no saturar el servidor
        # Puedes reducir más max_workers si sigue el problema
        with ThreadPoolExecutor(max_workers=2) as executor:
            for url in urls:
                executor.submit(download_image, url, images_dir)
    else:
        print(f"❌ No se encontró el archivo de URLs: {urls_file}")


Se han truncado las últimas 5000 líneas del flujo de salida.
✅ Descargada y válida: DNxXh6v.jpg
✅ Descargada y válida: DP11J2k.jpg
✅ Descargada y válida: DPEoWPK.jpg
✅ Descargada y válida: DPKF2ks.jpg
✅ Descargada y válida: DPtU0.jpg
✅ Descargada y válida: DQQZXBI.jpg
✅ Descargada y válida: DQxOadZ.jpg
✅ Descargada y válida: DRtcaWD.jpg
✅ Descargada y válida: DSRwIY0.jpg
✅ Descargada y válida: DSUSTJx.jpg
✅ Descargada y válida: DSmQZMX.jpg
🗑 Imagen descargada inválida (tamaño 161x81) → Eliminada: DSziBVv.jpg
✅ Descargada y válida: DTCtCcw.jpg
✅ Descargada y válida: DTGhzST.jpg
✅ Descargada y válida: DTsZ90e.jpg
✅ Descargada y válida: DVjbKtQ.jpg
✅ Descargada y válida: DX5k9cf.jpg
✅ Descargada y válida: DYiWlUN.jpg
✅ Descargada y válida: DYl7yQY.jpg
✅ Descargada y válida: DZKFLUa.jpg
✅ Descargada y válida: DZwUWAX.jpg
✅ Descargada y válida: DatP4Bq.jpg
✅ Descargada y válida: DauCiNi.jpg
✅ Descargada y válida: DaxGb6Z.jpg
✅ Descargada y válida: DbFp3Tx.jpg
✅ Descargada y válida: DbqLH6u.

## Borrar imagenes con dimension 161x81

In [ ]:
# # Dimensiones inválidas conocidas
# invalid_size = (161, 81)
# for cname in class_names:
#   # Ruta a la carpeta limpiar
#   folder_path = Path(f"/content/drive/MyDrive/LimpIA/data/{cname}/img")
#   # Recorremos todas las imágenes de la carpeta
#   for image_file in folder_path.glob("*"):
#       try:
#           with Image.open(image_file) as img:
#               if img.size == invalid_size:
#                   print(f"🗑 Eliminando: {image_file.name} — tamaño {img.size}")
#                   image_file.unlink()
#       except UnidentifiedImageError:
#           print(f"⚠️ No se pudo abrir: {image_file.name} (posiblemente corrupta)")
#           image_file.unlink()
#       except Exception as e:
#           print(f"⚠️ Error con {image_file.name}: {e}")
